# 步驟六：模型評估（Evaluation）執行指南

在完成隨機森林、XGBoost、DNN 三個模型的訓練後，評估階段的核心目標是：驗證模型在未見過數據（驗證集/測試集）上的泛化能力，並挑選出最能平衡「公司授信收益」與「違約損失」的模型與決策門檻。

為了落實「偽陰性代價極高」的業務考量，我們必須先評估模型預測機率的品質，並在計算分類指標前，先確定最佳決策門檻值。

以下是具體執行步驟與思考框架：

## 步驟 1：檢視預測機率的分桶違約率（機率校準度 Calibration）

許多人在評估時會直接調用模型的預測類別（$0$ 或 $1$），但在信用風險（先買後付）預測中，預測機率（Probability Scores）的真實物理意義才是靈魂。我們必須檢驗模型預測出的機率，是否與現實中的違約比例相符。

做法：

分別對 訓練集 (Training set) 與 測試集/驗證集 (Test/Validation set) 輸出模型預測的「用戶違約機率值」。

將預測機率依序等距劃分，每隔 10% 作為一個分桶區間（即：0~10%、11~20%、21~30% ... 91~100%）。

計算每個機率區間內的「實際違約人數佔比」（也就是統計該區間內 default_flag = 1 的比例）。

評估重點：

機率誠實度：如果模型將一群人的預測機率落在 11~20% 區間，那這群人的實際違約率是否真的接近 15% 左右？（若實際違約率高達 40%，代表模型嚴重「低估」風險，在先買後付場景中非常危險）。

過擬合/泛化能力檢查：對比訓練集與測試集在相同分桶下的違約比例是否一致。如果訓練集的機率校準很完美，但測試集在高機率區間的實際違約率卻發生大幅崩跌，代表模型發生了嚴重的過擬合。

## 步驟 2：繪製「雙曲線」評估模型整體的排序與區分能力

在調整任何決策門檻之前，必須先評估模型「區分好壞客戶」的本質能力（不依賴單一門檻）。請繪製以下兩張圖表：

### 1. ROC 曲線與 AUC (Area Under ROC)

原理：以 $False\ Positive\ Rate$（橫軸）與 $True\ Positive\ Rate\ (Recall)$（縱軸）繪製的曲線。

評估點：AUC 越接近 $1.0$ 代表模型對「準時付款」與「違約」的排序能力越好（即能把潛在違約者排在機率較高的位置）。

注意：ROC 曲線在類別不平衡（如違約率僅 $3\%$ ~ $5\%$）時容易顯得過於樂觀。

### 2. PR (Precision-Recall) 曲線與 AUC-PR

原理：以 $Recall$（橫軸）與 $Precision$（縱軸）繪製的曲線。

評估點：在不平衡數據中，PR 曲線更能真實反映模型在「違約（少數類別）」上的預測品質。

判定標準：曲線越往右上方靠近，代表模型越優秀。請優先選擇 PR 曲線包圍面積（AUC-PR）最大的模型。

## 步驟 3：關鍵優化——尋找與設定「最佳決策門檻值」

得到模型輸出的機率後，我們需要訂定一個恰當的門檻值 $t$（當預測機率 $\ge t$ 時，判定為違約 $1$；反之為 $0$）。以下提供三種在金融風控中最常用的量化設定方法：

最大化 $F_2$-Score 因為你特別重視召回率（Recall），利用訓練集、測試集計算不同門檻下的 $F_2$-Score，並選擇分數最高的點作為門檻。

$$F_2 = 5 \times \frac{Precision \times Recall}{4 \times Precision + Recall}$$

這會給予 Recall 兩倍於 Precision 的權重，強迫門檻往低處走，主動抓出更多潛在違約者。

## 步驟 4：計算不同門檻下的基本分類指標

在確定了步驟 3 的「最佳門檻 $t_{opt}$」後，我們才能將連續的預測機率轉化為二分類標籤（$0$ 或 $1$）。

請計算各模型在「預設門檻 0.5」與「最佳門檻 $t_{opt}$」兩種設定下的指標，進行對比呈現：

### 1. 混淆矩陣 (Confusion Matrix)：

統計出：真陽性 (TP)、真陰性 (TN)、偽陽性 (FP)、偽陰性 (FN)。

請特別指出並對比優化前後的 FN（漏報違約） 數量。

### 2. 準確率 (Accuracy)：

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

注意：僅供參考。若違約率僅 $5\%$，瞎猜所有人不違約便能得到 $95\%$ 準確率，因此該指標無法反映風控真實價值。

### 3. 精準度 (Precision)：

$$Precision = \frac{TP}{TP + FP}$$

商業意義：在模型預測「會違約」的人群中，實際上真的違約的比例。代表風控審核的精準度（避免誤殺好人，維持客戶體驗）。

### 4. 召回率 (Recall / Sensitivity)：

$$Recall = \frac{TP}{TP + FN}$$

商業意義：在所有實際違約的客戶中，模型成功抓出了多少人。這是你最需要提高的指標（FN 越低，Recall 越高）。

### 5. $F_2$-Score：

$$F_2 = 5 \times \frac{Precision \times Recall}{4 \times Precision + Recall}$$

商業意義：比起平衡的 $F_1$-Score，它更重視 Recall。用來評估模型在偏向風控安全時的綜合表現。